In [ ]:
## GRU with 10 Seeds

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
    !pip install -q pytorch_forecasting
else:
    PROJECT_ROOT = Path('../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

IN_COLAB=False  REPO_ROOT=/Users/ucabtt4/Projects/Dissertation/dissertation


In [2]:
import json

import numpy as np

from src.dataloader import load_cgm_stats, load_tensors, load_participant_ids
from src import metrics
from src import train
from src.models.nn_baseline import GRUPredictor, HORIZONS

MODEL_NAME = 'GRU'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '1_baseline'
RESULTS_DIR = EXPERIMENT_DIR / 'results'
STABILITY_SUMMARY_PATH = RESULTS_DIR / 'gru_stability_summary.json'

In [12]:
stats = load_cgm_stats()
train_loader = load_tensors('train')  # batch_size=256 default (Training Procedure)
val_loader = load_tensors('val')
test_loader = load_tensors('test')

# Unshuffled (default for 'test'), so this lines up row-for-row with train.predict(model,
# test_loader)'s output -- needed for the participant-level RMSE breakdown below, not just the
# pooled-across-all-test-samples numbers evaluate() reports on its own.
test_participant_ids = load_participant_ids('test')
PARTICIPANT_IDS = sorted(set(test_participant_ids))

print(f'number of train batches: ', len(train_loader), '\nnumber of valuation batches: ', len(val_loader),
      '\nnumber of test batches: ', len(test_loader))
print('number of test participants:', len(PARTICIPANT_IDS))
print('Corresponding participants ID: \n', PARTICIPANT_IDS)

sample_cgm, sample_target = next(iter(train_loader))
print('\ninput batch shape (cgm):', tuple(sample_cgm.shape))
print('target batch shape:', tuple(sample_target.shape))

number of train batches:  327 
number of valuation batches:  66 
number of test batches:  66
number of test participants: 6
Corresponding participants ID: 
 [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]

input batch shape (cgm): (256, 24)
target batch shape: (256, 5)


In [14]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []  # one entry per seed, in memory -- also written to disk individually below

for seed in SEEDS:
    print(f"\n{'-' * 32}seed={seed}{'-' * 32}")  # printed before training starts -- so it's
                                                    # visually obvious a new seed has begun, not
                                                    # that the notebook has hung
    train.set_seed(seed)  # before constructing the model -- weight init draws from this
    model = GRUPredictor()

    model, history = train.train_model(
        model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
        checkpoint_dir=EXPERIMENT_DIR / 'checkpoints' / f'gru_seed{seed}',
        verbose=False,  # quiet per-epoch text logging -- the progress bar below (and the one
                         # summary line per seed, after) carry that job instead
        progress_bar=True, progress_desc=f'seed={seed}',
    )
    n_epochs = len(history['train_loss'])

    y_pred_val, y_true_val = train.predict(model, val_loader)
    y_pred_test, y_true_test = train.predict(model, test_loader)
    val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
    test_results = metrics.evaluate(y_pred_test, y_true_test, stats['cgm_mean'], stats['cgm_std'])

    seed_result = {
        'model': MODEL_NAME, 'seed': seed, 'n_epochs_trained': n_epochs,
        'val_results': val_results, 'test_results': test_results,
    }
    all_results.append(seed_result)

    with open(RESULTS_DIR / f'gru_seed{seed}_results.json', 'w') as f:
        json.dump(seed_result, f, indent=2)

    test_rmse = [round(test_results[h]['rmse'], 2) for h in HORIZONS]
    print(f"seed={seed:>3}  stopped at epoch {n_epochs:>3}/50  "
          f"test RMSE @ {HORIZONS} min = {test_rmse}")


--------------------------------seed=7--------------------------------


seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

seed=  7  stopped at epoch  42/50  test RMSE @ [15, 30, 60, 90, 120] min = [11.49, 18.65, 27.53, 32.93, 36.03]

--------------------------------seed=14--------------------------------


seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

seed= 14  stopped at epoch  39/50  test RMSE @ [15, 30, 60, 90, 120] min = [11.24, 18.58, 27.58, 33.15, 36.49]

--------------------------------seed=21--------------------------------


...

seed= 56  stopped at epoch  31/50  test RMSE @ [15, 30, 60, 90, 120] min = [11.18, 18.46, 27.19, 32.43, 35.56]

--------------------------------seed=63--------------------------------


seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

seed= 63  stopped at epoch  36/50  test RMSE @ [15, 30, 60, 90, 120] min = [11.26, 18.5, 27.29, 32.72, 35.98]

--------------------------------seed=70--------------------------------


seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

seed= 70  stopped at epoch  44/50  test RMSE @ [15, 30, 60, 90, 120] min = [11.18, 18.44, 27.29, 32.69, 35.78]


In [ ]:
# Mean +/- SD across all len(SEEDS) seeds, per horizon, for every metric -- computed on the
# TEST split (the number that would actually be reported). Zone A and Zone A+B are both kept
# (not just A+B): A+B is the ISO 15197-comparable regulatory figure, A alone is the stricter,
# more discriminative one -- A+B is close to saturated (>99.7%) across every model in this
# screen, so it barely distinguishes between seeds either.
summary = {}
for h in HORIZONS:
    per_horizon = {}
    for m in ['rmse', 'mae', 'grmse']:
        values = [r['test_results'][h][m] for r in all_results]
        per_horizon[m] = {'mean': float(np.mean(values)), 'std': float(np.std(values))}
    for zone_label, zones in [('zone_a', ('A',)), ('zone_ab', ('A', 'B'))]:
        values = [sum(r['test_results'][h]['ceg'][z] for z in zones) for r in all_results]
        per_horizon[zone_label] = {'mean': float(np.mean(values)), 'std': float(np.std(values))}
    summary[h] = per_horizon

print(f'=== {MODEL_NAME} test-set stability across {len(SEEDS)} seeds ({SEEDS}) ===')
for h in HORIZONS:
    s = summary[h]
    print(f"{h:>3}-min:  RMSE = {s['rmse']['mean']:.1f} +/- {s['rmse']['std']:.1f} mg/dL   "
          f"MAE = {s['mae']['mean']:.1f} +/- {s['mae']['std']:.1f}   "
          f"gRMSE = {s['grmse']['mean']:.1f} +/- {s['grmse']['std']:.1f}   "
          f"Zone A = {s['zone_a']['mean']:.2f} +/- {s['zone_a']['std']:.2f}%   "
          f"Zone A+B = {s['zone_ab']['mean']:.2f} +/- {s['zone_ab']['std']:.2f}%")

with open(STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({'model': MODEL_NAME, 'seeds': SEEDS, 'n_seeds': len(SEEDS),
               'per_horizon_summary': summary}, f, indent=2)

print(f'\nsaved per-seed results to {RESULTS_DIR}/gru_seed*_results.json')
print(f'saved aggregate summary to {STABILITY_SUMMARY_PATH}')